# 06 – Evaluation: U-Net Lane Segmentation

**CSE445 – Road Damage Detection & Lane Segmentation**

This notebook:
1. Loads the best U-Net checkpoint from `experiments/lane_unet_run1/`
2. Runs inference on the held-out test set
3. Computes IoU, Dice, Pixel Accuracy, Precision, Recall
4. Visualises lane predictions on raw road frames
5. Analyses failure cases

**Prerequisites**: Run `04_train_lane_unet.ipynb` first.

In [ ]:
import sys, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/Road_Damage_Project'
    os.environ['RUN_ENV'] = 'colab'
except ImportError:
    REPO = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
    os.environ['RUN_ENV'] = 'local'

if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('Repo root:', REPO)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm

import config
from src.shared.dataset    import SegmentationDataset
from src.shared.transforms import get_val_transforms
from src.shared.unet       import UNet
from src.shared.metrics    import compute_all_metrics, iou_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

mean_arr = np.array([0.485, 0.456, 0.406])
std_arr  = np.array([0.229, 0.224, 0.225])

## 1. Load Model Checkpoint

In [ ]:
cfg  = config.LANE_UNET
ckpt = config.EXP_DIR / cfg['run_name'] / 'best_model.pth'

model = UNet(cfg['in_channels'], cfg['out_channels'], cfg['base_features'])
model.load_state_dict(torch.load(ckpt, map_location=device))
model = model.to(device).eval()
print('Loaded:', ckpt)

## 2. Test DataLoader

In [ ]:
test_ds = SegmentationDataset(
    config.LANE_SPLIT_DIR / 'test.csv',
    transform=get_val_transforms((config.IMG_HEIGHT, config.IMG_WIDTH))
)
test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=2)
print(f'Test set: {len(test_ds)} samples')

## 3. Quantitative Evaluation

In [ ]:
all_iou, all_dice, all_acc, all_prec, all_rec = [], [], [], [], []

with torch.no_grad():
    for images, masks in tqdm(test_loader, desc='Evaluating'):
        images = images.to(device)
        masks  = masks.to(device)
        preds  = model(images)
        m      = compute_all_metrics(preds, masks)
        all_iou.append(m['iou'])
        all_dice.append(m['dice'])
        all_acc.append(m['pixel_acc'])
        all_prec.append(m['precision'])
        all_rec.append(m['recall'])

mean_iou  = np.mean(all_iou)
mean_dice = np.mean(all_dice)
mean_acc  = np.mean(all_acc)
mean_prec = np.mean(all_prec)
mean_rec  = np.mean(all_rec)

print('\n' + '='*50)
print('  LANE SEGMENTATION – TEST SET RESULTS')
print('='*50)
print(f'  IoU (Jaccard)  : {mean_iou:.4f}')
print(f'  Dice (F1)      : {mean_dice:.4f}')
print(f'  Pixel Accuracy : {mean_acc:.4f}')
print(f'  Precision      : {mean_prec:.4f}')
print(f'  Recall         : {mean_rec:.4f}')
print('='*50)

fig, ax = plt.subplots(figsize=(8, 4))
metrics = ['IoU', 'Dice', 'Pixel Acc', 'Precision', 'Recall']
values  = [mean_iou, mean_dice, mean_acc, mean_prec, mean_rec]
colors  = ['#F57C00', '#E64A19', '#6A1B9A', '#388E3C', '#C2185B']
bars = ax.bar(metrics, values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_title('U-Net Lane Segmentation – Test Set Metrics')
plt.tight_layout(); plt.show()

## 4. Visual Predictions (Yellow Lane Overlay)

In [ ]:
N = 6
fig, axes = plt.subplots(N, 4, figsize=(16, N * 3))
fig.suptitle('Lane Segmentation – Predictions vs Ground Truth', fontsize=12)

batch_imgs, batch_masks = next(iter(DataLoader(test_ds, batch_size=N, shuffle=True)))
with torch.no_grad():
    batch_preds = model(batch_imgs.to(device)).cpu()

for i in range(N):
    img_np    = (batch_imgs[i].permute(1,2,0).numpy() * std_arr + mean_arr).clip(0, 1)
    img_uint8 = (img_np * 255).astype(np.uint8)
    mask_np   = batch_masks[i].squeeze().numpy()
    pred_np   = (batch_preds[i].squeeze().numpy() > 0.5).astype(np.uint8)

    # Yellow GT overlay
    gt_overlay = img_uint8.copy()
    gt_overlay[mask_np > 0.5] = [255, 220, 0]

    # Cyan prediction overlay
    pred_overlay = img_uint8.copy()
    pred_overlay[pred_np > 0] = [0, 220, 220]

    axes[i, 0].imshow(img_np);           axes[i, 0].set_title('Frame')
    axes[i, 1].imshow(gt_overlay);       axes[i, 1].set_title('GT Overlay (yellow)')
    axes[i, 2].imshow(pred_overlay);     axes[i, 2].set_title('Pred Overlay (cyan)')
    axes[i, 3].imshow(mask_np, cmap='gray'); axes[i, 3].set_title('GT Mask')
    for ax in axes[i]: ax.axis('off')

plt.tight_layout(); plt.show()

## 5. Failure Case Analysis

In [ ]:
sample_ious = []
with torch.no_grad():
    for images, masks in test_loader:
        preds = model(images.to(device)).cpu()
        for i in range(preds.shape[0]):
            sample_ious.append(iou_score(preds[i:i+1], masks[i:i+1]))

sample_ious = np.array(sample_ious)
worst_idx   = np.argsort(sample_ious)[:4]

print('Worst IoU scores:', sample_ious[worst_idx])

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('Failure Cases – Lowest IoU Lane Predictions', fontsize=12)

for col, idx in enumerate(worst_idx):
    img_t, mask_t = test_ds[idx]
    with torch.no_grad():
        pred_t = model(img_t.unsqueeze(0).to(device)).squeeze().cpu()

    img_np   = (img_t.permute(1,2,0).numpy() * std_arr + mean_arr).clip(0, 1)
    pred_np  = (pred_t.numpy() > 0.5).astype(np.float32)
    mask_np  = mask_t.squeeze().numpy()

    axes[0, col].imshow(img_np);  axes[0, col].set_title(f'Frame (IoU={sample_ious[idx]:.3f})')
    axes[1, col].imshow(np.stack([pred_np, mask_np, np.zeros_like(pred_np)], axis=-1))
    axes[1, col].set_title('Red=Pred  Green=GT')
    for row in range(2): axes[row, col].axis('off')

plt.tight_layout(); plt.show()

## Summary

Record your results here:

| Model | IoU | Dice | Pixel Acc | Precision | Recall |
|---|---|---|---|---|---|
| U-Net Lane (run1) | __ | __ | __ | __ | __ |

### Observations
- Failure cases often occur at scene boundaries, shadows, or lane merges
- Consider increasing training image resolution or adding GaussianBlur augmentation
- TuSimple IoU ≥ 0.70 is considered good performance in literature